# Jupyter Notebook Radar Inference

Run inference of a pre-trained radar physiologic (resp) deep model on the CogPhys test set, compute resp rate (RR), evaluate errors, and save predictions.

In [ ]:
# autorelead libraries
%load_ext autoreload
%autoreload 2

## Import all libraries

In [ ]:
import os
import time
import random
import pickle
import argparse

import numpy as np
import scipy.sparse as sp
import scipy.signal as sig
import matplotlib.pyplot as plt
from tqdm import tqdm
from copy import deepcopy

import torch
from torch.utils.data import DataLoader

from config import get_config
from dataset import data_loader
from neural_methods.model.RadarNet import RadarNet

For Multi GPU systems, choose GPU number

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Essential Functions

- Detrending function to remove unwanted signal trends in the estiamtion
- Function to estimate the resp rate from the power spectral density. For resp, it's the regular Periodogram
- Functions to calcualte metrics such as the SNR and MACC

In [ ]:
def custom_detrend(sig, Lambda):
    """custom_detrend(sig, Lambda) -> filtered_signal
    This function applies a detrending filter.
    This code is based on the following article "An advanced detrending method with application
    to HRV analysis". Tarvainen et al., IEEE Trans on Biomedical Engineering, 2002.
    *Parameters*
      ``sig`` (1d numpy array):
        The sig where you want to remove the trend.
      ``Lambda`` (int):
        The smoothing parameter.
    *Returns*
      ``filtered_signal`` (1d numpy array):
        The detrended sig.
    """
    signal_length = sig.shape[0]

    # observation matrix
    H = np.identity(signal_length)

    # second-order difference matrix

    ones = np.ones(signal_length)
    minus_twos = -2 * np.ones(signal_length)
    diags_data = np.array([ones, minus_twos, ones])
    diags_index = np.array([0, 1, 2])
    D = sp.spdiags(diags_data, diags_index, (signal_length - 2), signal_length).toarray()
    filtered_signal = np.dot((H - np.linalg.inv(H + (Lambda ** 2) * np.dot(D.T, D))), sig)
    return filtered_signal

def resp_rate_from_power_spectral_density(pleth_sig: np.array, FS: float,
                                           LL_PR: float, UL_PR: float,
                                           BUTTER_ORDER: int = 6,
                                           DETREND: bool = False,
                                           FResBPM: float = 0.1,
                                           HARMONIC: bool = False,
                                           WELCH = True) -> float:
    """ Function to estimate the pulse rate from the power spectral density of the plethysmography sig.

    Args:
        pleth_sig (np.array): Plethysmography sig.
        FS (float): Sampling frequency.
        LL_PR (float): Lower cutoff frequency for the butterworth filtering.
        UL_PR (float): Upper cutoff frequency for the butterworth filtering.
        BUTTER_ORDER (int, optional): Order of the butterworth filter. Give None to skip filtering. Defaults to 6.
        DETREND (bool, optional): Boolena Flag for executing cutsom_detrend. Defaults to False.
        FResBPM (float, optional): Frequency resolution. Defaults to 0.1.

    Returns:
        pulse_rate (float): _description_
    

    Daniel McDuff, Ethan Blackford, January 2019
    Copyright (c)
    Licensed under the MIT License and the RAIL AI License.
    """

    N = (60*FS)/FResBPM

    # Detrending + nth order butterworth + periodogram
    if DETREND:
        pleth_sig = custom_detrend(pleth_sig, 100)
    if BUTTER_ORDER:
        [b, a] = sig.butter(BUTTER_ORDER, [LL_PR/60, UL_PR/60], btype='bandpass', fs = FS)
    pleth_sig = sig.filtfilt(b, a, np.double(pleth_sig))
    
    # Calculate the PSD and the mask for the desired range
    if WELCH:
        F, Pxx = sig.welch(x=pleth_sig, nperseg=len(pleth_sig)//3, nfft=N, fs=FS)
    else:
        F, Pxx = sig.periodogram(x=pleth_sig,  nfft=N, fs=FS);  
    FMask = (F >= (LL_PR/60)) & (F <= (UL_PR/60))
    
    # Calculate predicted pulse rate:
    FRange = F * FMask
    PRange = Pxx * FMask

    if HARMONIC:
      harmonicsigPower = PRange[::2]
      harmonicsigPower = np.pad(harmonicsigPower, (0, len(PRange) - len(harmonicsigPower)))
      harmonicsigPower[0] = 0
      MaxInd = np.argmax(PRange+harmonicsigPower)
    else:
      MaxInd = np.argmax(PRange)
    pulse_rate_freq = FRange[MaxInd]
    pulse_rate = pulse_rate_freq*60
            
    return pulse_rate

def power2db(mag):
    """Convert power to db."""
    return 10 * np.log10(mag)

def _next_power_of_2(x):
    """Calculate the nearest power of 2."""
    return 1 if x == 0 else 2 ** (x - 1).bit_length()

def compute_macc(pred_signal, gt_signal):
    """Calculate maximum amplitude of cross correlation (MACC) by computing correlation at all time lags.
        Args:
            pred_ppg_signal(np.array): predicted PPG signal 
            label_ppg_signal(np.array): ground truth, label PPG signal
        Returns:
            MACC(float): Maximum Amplitude of Cross-Correlation
    """
    pred = deepcopy(pred_signal)
    gt = deepcopy(gt_signal)
    pred = np.squeeze(pred)
    gt = np.squeeze(gt)
    min_len = np.min((len(pred), len(gt)))
    pred = pred[:min_len]
    gt = gt[:min_len]
    lags = np.arange(0, len(pred)-1, 1)
    tlcc_list = []
    for lag in lags:
        cross_corr = np.abs(np.corrcoef(
            pred, np.roll(gt, lag))[0][1])
        tlcc_list.append(cross_corr)
    macc = max(tlcc_list)
    return macc

def calculate_SNR(pred_ppg_signal, hr_label, fs=30, low_pass=0.6, high_pass=3.3):
    """Calculate SNR as the ratio of the area under the curve of the frequency spectrum around the first and second harmonics 
        of the ground truth HR frequency to the area under the curve of the remainder of the frequency spectrum, from 0.6 Hz
        to 3.3 Hz. 

        Ref for low_pass and high_pass filters:
        R. Cassani, A. Tiwari and T. H. Falk, "Optimal filter characterization for photoplethysmography-based pulse rate and 
        pulse power spectrum estimation," 2020 IEEE Engineering in Medicine & Biology Society (EMBC), Montreal, QC, Canada,
        doi: 10.1109/EMBC44109.2020.9175396.

        Note: to more closely match results in the NeurIPS 2023 toolbox paper, we recommend low_pass=0.75 and high_pass=2.5 
        instead of the defaults above.

        Args:
            pred_ppg_signal(np.array): predicted PPG signal 
            label_ppg_signal(np.array): ground truth, label PPG signal
            fs(int or float): sampling rate of the video
        Returns:
            SNR(float): Signal-to-Noise Ratio
    """
    # Get the first and second harmonics of the ground truth HR in Hz
    first_harmonic_freq = hr_label / 60
    second_harmonic_freq = 2 * first_harmonic_freq
    deviation = 6 / 60  # 6 beats/min converted to Hz (1 Hz = 60 beats/min)

    # Calculate FFT
    pred_ppg_signal = np.expand_dims(pred_ppg_signal, 0)
    N = _next_power_of_2(pred_ppg_signal.shape[1])
    f_ppg, pxx_ppg = sig.periodogram(pred_ppg_signal, fs=fs, nfft=N, detrend=False)

    # Calculate the indices corresponding to the frequency ranges
    idx_harmonic1 = np.argwhere((f_ppg >= (first_harmonic_freq - deviation)) & (f_ppg <= (first_harmonic_freq + deviation)))
    idx_harmonic2 = np.argwhere((f_ppg >= (second_harmonic_freq - deviation)) & (f_ppg <= (second_harmonic_freq + deviation)))
    idx_remainder = np.argwhere((f_ppg >= low_pass) & (f_ppg <= high_pass) \
     & ~((f_ppg >= (first_harmonic_freq - deviation)) & (f_ppg <= (first_harmonic_freq + deviation))) \
     & ~((f_ppg >= (second_harmonic_freq - deviation)) & (f_ppg <= (second_harmonic_freq + deviation))))

    # Select the corresponding values from the periodogram
    pxx_ppg = np.squeeze(pxx_ppg)
    pxx_harmonic1 = pxx_ppg[idx_harmonic1]
    pxx_harmonic2 = pxx_ppg[idx_harmonic2]
    pxx_remainder = pxx_ppg[idx_remainder]

    # Calculate the signal power
    signal_power_hm1 = np.sum(pxx_harmonic1)
    signal_power_hm2 = np.sum(pxx_harmonic2)
    signal_power_rem = np.sum(pxx_remainder)

    # Calculate the SNR as the ratio of the areas
    if not signal_power_rem == 0: # catches divide by 0 runtime warning 
        SNR = power2db((signal_power_hm1 + signal_power_hm2) / signal_power_rem)
    else:
        SNR = 0
    return SNR

## Load the Config and Test dataloader

In [ ]:
class Args:
    config_file = 'configs/train_configs/CogPhys_Resp_Radar_BASIC.yaml'
    cached_path = None
    preprocess = None
    lr = None
    model_file_name = None

args = Args()
config = get_config(args)
# print('Configuration:')
# print(config, end='\n\n')

test_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
test_data_loader = test_loader(
    name="test",
    data_path=config.TEST.DATA.DATA_PATH,
    config_data=config.TEST.DATA,
    device=config.DEVICE)

In [ ]:
fold_num = config.TEST.DATA.FOLD.FOLD_NAME
print(fold_num)

In [ ]:
test_data_loader.input_preproc, test_data_loader.label_preproc, test_data_loader.input_keys, test_data_loader.label_keys

In [ ]:
torch.cuda.empty_cache()

## Load the Models and Create a Forward Pass Function

In [ ]:
channels = config.MODEL.RADARNET.CHANNELS
print("RadarNet channels: ", channels)

In [ ]:
load_path = ""
model = RadarNet(channels=channels).to(config.DEVICE).eval()
model.load_state_dict(torch.load(load_path))

In [ ]:
all_pred = []
all_gt = []
all_participant_task_chunk_list = []
for i in range(0, len(test_data_loader), 6):
    for j in [[0, 1], [2, 3], [4, 5]]:
        radar_matrix = []
        label = []
        participant_task_list = []
        chunk_id_list = []
        with torch.no_grad():
            for k in j:
                radar_sample, label_sample, participant_task, chunk_id = test_data_loader[i+k]
                participant_task_list.append(participant_task)
                chunk_id_list.append(int(chunk_id))
                radar_matrix.append(radar_sample.to(config.DEVICE))
                label.extend(label_sample.squeeze(0).cpu().numpy().tolist())
            radar_matrix = torch.cat(radar_matrix, dim=0).unsqueeze(0)[...,:channels//2].permute(0, 2, 3, 1)
            radar_matrix = radar_matrix.reshape(radar_matrix.shape[0], -1, radar_matrix.shape[3])
            pred = model(radar_matrix)[0].squeeze(0).cpu().numpy()
        ##################
        assert participant_task_list[0] == participant_task_list[1]
        assert chunk_id_list[0] == chunk_id_list[1]-1
        ##################
        pred = np.array(pred)
        label = np.array(label)
        all_participant_task_chunk_list.append((participant_task_list[0], chunk_id_list)) 
        all_pred.append(pred)
        all_gt.append(label)
        print(participant_task_list[0], chunk_id_list)

## Process the waveforms and Calculate the Vitals

The waveforms are normalized, detrended and filtered before vitals calculation.
- RR: Heart rate from the resp and remote resp signals. Calculated using the periodogram

We also calculate two waveform metrics:

- SNR: Signal to Noise Ratio
- MACC: Maximum Amplitude of Cross-Correlation

In [ ]:
fs = config.TEST.DATA.FS
ll_cutoff = 8
ul_cutoff = 30
print(fs, ll_cutoff, ul_cutoff)
all_pred_rr = []
all_gt_rr = []
all_macc = []
all_snr = []
for pred, gt, (participant_task, chunk_id_list) in zip(all_pred, all_gt, all_participant_task_chunk_list):
    print(participant_task, chunk_id_list)
    # Norm
    pred = (pred - np.mean(pred)) / np.std(pred)
    gt = (gt - np.mean(gt)) / np.std(gt)
    # 1-D gauss blur
    pred = np.convolve(pred, np.ones((15))/15, mode='same')
    gt = np.convolve(gt, np.ones((15))/15, mode='same')
    # Norm
    pred = (pred - np.mean(pred)) / np.std(pred)
    gt = (gt - np.mean(gt)) / np.std(gt)
    # Calculate RR
    pred_rr = resp_rate_from_power_spectral_density(pred, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=2, 
                                                        DETREND=True, WELCH=False)
    gt_rr = resp_rate_from_power_spectral_density(gt, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=2, 
                                                        DETREND=True, WELCH=False)
    all_pred_rr.append(pred_rr)
    all_gt_rr.append(gt_rr)
    # Calculate MACC
    all_macc.append(compute_macc(pred, gt))
    # Calculate SNR
    all_snr.append(calculate_SNR(pred, gt_rr, fs=fs, low_pass=0.1, high_pass=0.6))
    print(f"| {pred_rr:.2f} - {gt_rr:.2f} | = {abs(pred_rr - gt_rr):.2f} bpm")
    print("-"*100)

all_pred_rr = np.array(all_pred_rr)
all_gt_rr = np.array(all_gt_rr)
# Remove nan values from SNR and MACC lists
all_snr = np.array([snr for snr in all_snr if not np.isnan(snr)])
all_macc = np.array([macc for macc in all_macc if not np.isnan(macc)])

## Calculate the error

The following error values are calcualted (optional with their std error spread)
- MAE: Mean Absolute Error
- RMSE: Root Mean Square Error
- MAPE: Mean Absolute Percentage Error
- r: Pearson Correlation Coefficient

In [ ]:
def get_error_metric(pred_values, gt_values, return_se=False):
    """
    Calculate the error metric between predicted and ground truth values.
    Returns means and optionally standard errors.
    """
    num_samples = len(pred_values)
    
    # Calculate the mean absolute error
    abs_errors = np.abs(pred_values - gt_values)
    mae = np.mean(abs_errors)
    mae_se = np.std(abs_errors) / np.sqrt(num_samples) if return_se else None
    
    # Calculate the root mean squared error
    squared_errors = np.square(pred_values - gt_values)
    rmse = np.sqrt(np.mean(squared_errors))
    rmse_se = np.sqrt(np.std(squared_errors) / np.sqrt(num_samples)) if return_se else None
    
    # Calculate the mean absolute percentage error
    percentage_errors = np.abs((pred_values - gt_values) / gt_values)
    mape = np.mean(percentage_errors) * 100
    mape_se = np.std(percentage_errors) / np.sqrt(num_samples) * 100 if return_se else None
    
    # Calculate pearson correlation coefficient
    r = np.corrcoef(pred_values, gt_values)[0, 1]
    r_se = np.sqrt((1 - r**2) / (num_samples - 2)) if return_se else None
    
    if return_se:
        return (mae, mae_se), (rmse, rmse_se), (mape, mape_se), (r, r_se)
    else:
        return mae, rmse, mape, r

In [ ]:
# MAE, RMSE, MAPE, r
(mae, mae_std), (rmse, rmse_std), (mape, mape_std), (r, r_std) = get_error_metric(all_pred_rr, all_gt_rr, return_se=True)
snr_error = np.mean(all_snr)
snr_std = np.std(all_snr) / np.sqrt(len(all_snr))
macc_error = np.mean(all_macc)
macc_std = np.std(all_macc) / np.sqrt(len(all_macc))
# Round to 2 decimal places
print(f"MAE: {mae:.2f} ± {mae_std:.2f} bpm")
print(f"RMSE: {rmse:.2f} ± {rmse_std:.2f} bpm")
print(f"MAPE: {mape:.2f} ± {mape_std:.2f} %")
print(f"r: {r:.2f} ± {r_std:.2f}")
print(f"SNR: {snr_error:.2f} ± {snr_std:.2f} dB")
print(f"MACC: {macc_error:.2f} ± {macc_std:.2f}")

## Save the waveforms and vitals

In [ ]:
# Name of the save folder
print(len(all_pred), len(all_gt), len(all_participant_task_chunk_list))
save_folder = f"{fold_num}/{fold_num}_{'_'.join(config.TEST.DATA.COGPHYS.INPUT)}"
print(save_folder)
os.makedirs(save_folder, exist_ok=True)

# Save the waveforms
waveform_dict = {
    'pred': all_pred, 'gt': all_gt, 
    'participant_task_chunk_id_list': all_participant_task_chunk_list
}
with open(os.path.join(save_folder, "pred.pickle"), 'wb') as f:
    pickle.dump(waveform_dict, f)

# Save the vitals values as a pickle
output_data = {
    'pred_rr': all_pred_rr,
    'gt_rr': all_gt_rr,
    'snr': all_snr,
    'macc': all_macc,
    'participant_task_chunk_list': all_participant_task_chunk_list
}
with open(os.path.join(save_folder, "vitals.pickle"), 'wb') as f:
    pickle.dump(output_data, f)